<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/04_葡萄酒品種辨識_經典分類演算法篇/葡萄酒品種辨識_經典分類演算法篇.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
%%capture
# 在 Google Colab 執行時，先安裝中文字型與資料下載支援
%pip install wget mplfonts scipy seaborn


# 🍷 範例 04：葡萄酒品種辨識（經典分類演算法篇） — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本筆記本沿用範例 01 的 **「一段程式碼 + 一段 AI 互動學習指引」** 架構：

- 每個程式儲存格都提供可直接執行的資料處理、建模與評估流程。
- 每段程式碼的正下方都有白話說明與專屬 Prompt；可直接複製給 AI 追問原理或練習調整。
- 資料取自《資料科學自學聖經》的 `wine.csv`，在 Colab 找不到檔案時會自動下載。

請從步驟 0 依序執行。交叉驗證與調參只能使用訓練資料；測試集保留到最後一次驗收。


---  
### 【步驟 0】：環境準備與繁體中文字型設定
載入資料科學套件並設定中文字型，避免圖表顯示亂碼。


In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK TC")
except Exception:
    print("提示：中文字型設定失敗時，模型仍可正常執行。")
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")


#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

載入資料處理與視覺化套件後，先確認環境再開始模型實驗；中文字型失敗不會影響模型訓練。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---  
### 【步驟 1】：資料載入與結構探索（EDA）
讀取資料、查看欄位型態與摘要統計，先了解資料再下模型。


In [ ]:
# [步驟 1 程式碼] 資料載入與結構探索
if not os.path.exists("wine.csv"):
    import wget
    wget.download("https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/04_葡萄酒品種辨識_經典分類演算法篇/wine.csv", "wine.csv")
df = pd.read_csv("wine.csv")
print(f"資料筆數：{len(df):,}；欄位數：{df.shape[1]}")
display(df.head())
display(df.describe(include="all").T)
print("多類別分類；目標欄位：酒種類")


#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

本專案要學會：K-近鄰、決策樹、核化 SVM、高斯樸素貝氏、梯度提升與 Stacking。先檢查資料型態、缺失值、重複值與可疑欄位。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 2】：特徵關係與視覺化
先看類別分布及兩個特徵的關係；這是探索資料，不是用兩個特徵取代完整模型。


In [ ]:
# [步驟 2 程式碼] 類別分布與前兩個特徵
X = df.drop(columns="酒種類")
display(df["酒種類"].value_counts().sort_index().rename("筆數"))
cols = list(X.columns)
sns.scatterplot(data=df, x=cols[0], y=cols[1], hue="酒種類", palette="Set2", s=65)
plt.title("前兩個特徵的類別分布"); plt.show()


#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

散佈圖只能同時呈現兩個特徵；真正模型仍使用全部特徵。類別分布也會影響 Accuracy 的解讀。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 3】：資料切分、Pipeline 與交叉驗證
所有前處理都放進 Pipeline，並只在訓練資料中擬合，避免測試資料洩漏。


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report
X = df.drop(columns="酒種類")
y = df["酒種類"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
models = {
 "KNN": Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(5))]),
 "決策樹": DecisionTreeClassifier(max_depth=4, random_state=42),
 "核化 SVM": Pipeline([("scale", StandardScaler()), ("model", SVC(C=2, kernel="rbf"))]),
 "高斯貝氏": GaussianNB(),
 "梯度提升": GradientBoostingClassifier(random_state=42),
 "Stacking": StackingClassifier(estimators=[
  ("knn", Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(5))])),
  ("tree", DecisionTreeClassifier(max_depth=4, random_state=42)), ("nb", GaussianNB())],
  final_estimator=LogisticRegression(max_iter=2000), cv=5)}
# [步驟 3 程式碼] Pipeline 模型比較與 5 折交叉驗證
cv_results = {name: cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy").mean() for name, model in models.items()}
result_df = pd.Series(cv_results, name="5 折 Accuracy").sort_values(ascending=False).to_frame()
display(result_df.style.format("{:.3f}"))


#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

交叉驗證只在訓練資料內完成。KNN、SVM 和神經網路都需要標準化；樹模型通常不需要。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 4】：測試集大考與指標解讀
選定模型後才碰測試集。它的角色是最終驗收，不是反覆調參用的練習題。


In [ ]:
# [步驟 4 程式碼] 測試集大考
best_name = result_df.index[0]
best_model = models[best_name].fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(f"交叉驗證最佳模型：{best_name}")
print(f"測試集 Accuracy：{accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title(f"{best_name}：測試集混淆矩陣"); plt.show()


#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

高 Accuracy 不代表唯一最佳模型，也必須考慮可解釋性、速度與維護成本。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 5】：模型診斷與可解釋性
利用圖表了解模型為什麼這樣表現，再決定是否值得投入更多複雜度。


In [ ]:
# [步驟 5 程式碼] 模型診斷圖
plt.figure(figsize=(8, 4))
result_df["5 折 Accuracy"].sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("5 折交叉驗證 Accuracy"); plt.title("模型選型比較：不要只看單一次切分"); plt.show()


#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

診斷圖用來提出下一個問題，不是保證模型一定能用。請留意資料量、錯誤成本與公平性。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。


---
### 【步驟 6】：未見資料的實務推論
最後以一筆模型從未參與訓練的資料模擬部署時的預測流程。


In [ ]:
# [步驟 6 程式碼] 一筆未見資料的推論
new_case = X_test.iloc[[0]]
print("模型預測：", best_model.predict(new_case)[0])
print("實際標籤（課後核對）：", y_test.iloc[0])


#### 💬【步驟 6 學習指引與 AI 提問 Prompt】

這筆資料在模型訓練時未出現；請先猜測再看模型答案。真實應用還需要持續監控資料漂移。

> ❓ 請用生活化比喻解釋這一步的原理，並指出一個常見誤解。

> 🛠️ 請只改一個超參數，比較修改前後的結果；不要使用測試集選參數。
